# DXF Import — AutoCAD Layers to Gmsh Physical Groups

This notebook demonstrates loading an AutoCAD `.dxf` file into pyGmsh,
where **layers** are automatically mapped to **physical groups**, and then
generating a 1D transfinite mesh suitable for frame analysis.

The test file `frame_2D.dxf` is a 2-story structural frame with layers:
- `C80x80` — columns (80×80 section)
- `C40x40` — columns (40×40 section)
- `V30x50` — beams / vigas (30×50 section)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from pyGmsh import pyGmsh
import gmsh

## 1. Load the DXF file

In [ ]:
g = pyGmsh(verbose=True)
g.initialize()

layers = g.model.load_dxf("frame_2D.dxf")

n_points = len(gmsh.model.getEntities(0))
n_curves = len(gmsh.model.getEntities(1))
print(f"Points: {n_points}")
print(f"Curves: {n_curves}")

## 2. Inspect layers

In [ ]:
for name, dim_tags in layers.items():
    for dim, tags in dim_tags.items():
        print(f"Layer '{name}'  dim={dim}  n={len(tags)}  tags={tags}")

## 3. Verify physical groups

In [ ]:
for dim, pg_tag in gmsh.model.getPhysicalGroups(-1):
    name = gmsh.model.getPhysicalName(dim, pg_tag)
    ents = list(gmsh.model.getEntitiesForPhysicalGroup(dim, pg_tag))
    print(f"  dim={dim}  name='{name}'  entities={ents}")

## 4. Plot geometry with entity labels

Visualise the imported frame geometry using `g.plot.geometry()` with
entity tag annotations so we can verify each member's tag and layer.

In [ ]:
%matplotlib inline

g.plot.geometry(
    show_points=True,
    show_curves=True,
    show_surfaces=False,
    label_tags=True,
    show=True,
)

## 5. Transfinite mesh — uniform elements per member

Set a fixed number of elements for each structural member.
This gives a uniform discretisation along every column and beam.

In [ ]:
N_ELEM = 4  # elements per member
n_nodes = N_ELEM + 1  # transfinite needs node count, not element count

# Apply transfinite constraint to every curve in every layer
for layer_name, dim_tags in layers.items():
    for tag in dim_tags.get(1, []):
        g.mesh.set_transfinite_curve(tag, n_nodes)

print(f"Set {n_nodes} nodes ({N_ELEM} elements) on all {n_curves} members.")

## 6. Generate 1D mesh

In [ ]:
g.mesh.generate(1)

node_tags, _, _ = gmsh.model.mesh.getNodes()
elem_types, elem_tags, _ = gmsh.model.mesh.getElements(1)
n_mesh_nodes = len(node_tags)
n_mesh_elems = sum(len(et) for et in elem_tags)
print(f"Mesh nodes: {n_mesh_nodes}")
print(f"Mesh elements: {n_mesh_elems}  (expected {N_ELEM} x {n_curves} = {N_ELEM * n_curves})")

## 7. Plot the mesh

In [ ]:
g.plot.clear()
g.plot.mesh(show=True)

## 8. Entity registry

In [ ]:
g.model.registry()

## 9. Open the GUI (optional)

Uncomment to visually inspect in the Gmsh window.

In [ ]:
# g.model.gui()

## 10. Cleanup

In [ ]:
g.finalize()